# License
This notebook is licensed under the MIT License. See the LICENSE file in the repository root for details.

In [ ]:
# Setup for Colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !pip install -q pandas scikit-learn numpy Faker matplotlib seaborn
    print('Packages installed.')


# Recommender Systems In-Depth

Recommender systems are algorithms designed to suggest relevant items to users (e.g., movies to watch, books to read). They are fundamental to modern web applications.

In this notebook, we will explore two primary techniques using a synthetic dataset of **Books** and **User Ratings**:

1.  **Content-Based Filtering**: Recommends items similar to those a user liked, based on item attributes (e.g., Book Descriptions).
2.  **Collaborative Filtering**: Recommends items based on the past interactions of similar users (e.g., Matrix Factorization).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from faker import Faker
import random
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import mean_squared_error

fake = Faker()
Faker.seed(42)
np.random.seed(42)

sns.set_style('whitegrid')

## 1. Data Generation

We will generate:
*   **Books**: ID, Title, and Description (Genre/Keywords).
*   **Users**: ID and Name.
*   **Ratings**: User-Book interactions (1-5 stars).

In [ ]:
# Generate Books
n_books = 20
genres = ['Science Fiction', 'Romance', 'Mystery', 'History', 'Cooking']
books = []
for i in range(n_books):
    genre = random.choice(genres)
    books.append({
        'book_id': i,
        'title': fake.catch_phrase(),
        'genre': genre,
        'description': f"{genre} book about {fake.bs()} and {fake.word()}."
    })
df_books = pd.DataFrame(books)
print("--- Sample Books ---")
print(df_books[['book_id', 'title', 'genre']].head())

# Generate Ratings
n_users = 10
ratings = []
for user_id in range(n_users):
    # Each user rates 5-10 random books
    rated_books = random.sample(range(n_books), k=random.randint(5, 10))
    for book_id in rated_books:
        # Simulate preference: specific genres get higher ratings
        user_fav_genre = genres[user_id % len(genres)]
        book_genre = df_books.loc[book_id, 'genre']
        
        if book_genre == user_fav_genre:
            rating = random.randint(4, 5)
        else:
            rating = random.randint(1, 4)
            
        ratings.append({'user_id': user_id, 'book_id': book_id, 'rating': rating})

df_ratings = pd.DataFrame(ratings)
print("\n--- Sample Ratings ---")
print(df_ratings.head())

## 2. Content-Based Filtering

We recommend books similar to a specific book based on its text description. We use **TF-IDF** (Term Frequency-Inverse Document Frequency) to vectorize descriptions and **Cosine Similarity** to measure closeness.

In [ ]:
# Vectorize descriptions
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(df_books['description'])

# Compute Cosine Similarity Matrix
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

# Function to get recommendations
def get_content_recommendations(book_id, sim_matrix, n=3):
    # Get similarity scores for this book
    sim_scores = list(enumerate(sim_matrix[book_id]))
    # Sort by score (descending)
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    # Get top N (excluding itself)
    top_indices = [i[0] for i in sim_scores[1:n+1]]
    return df_books.iloc[top_indices]

target_book_id = 0
print(f"Recommendations for Book {target_book_id}: {df_books.iloc[target_book_id]['title']} ({df_books.iloc[target_book_id]['genre']})")
print(get_content_recommendations(target_book_id, cosine_sim))

## 3. Collaborative Filtering (Matrix Factorization)

We use **SVD (Singular Value Decomposition)** to decompose the User-Item matrix. This captures latent features (patterns) in user behavior.

In [ ]:
# Create User-Item Matrix (Pivot Table)
user_item_matrix = df_ratings.pivot(index='user_id', columns='book_id', values='rating').fillna(0)
print("User-Item Matrix Shape:", user_item_matrix.shape)

# Decompose matrix using Truncated SVD
svd = TruncatedSVD(n_components=5, random_state=42)
matrix_reduced = svd.fit_transform(user_item_matrix)
matrix_reconstructed = svd.inverse_transform(matrix_reduced)

# Convert back to DataFrame
df_reconstructed = pd.DataFrame(matrix_reconstructed, index=user_item_matrix.index, columns=user_item_matrix.columns)

print("\n--- Original Rating (User 0, Book 1) ---")
print(user_item_matrix.iloc[0, 1])
print("--- Predicted Rating (User 0, Book 1) ---")
print(df_reconstructed.iloc[0, 1])

## 4. Evaluation

We calculate the Root Mean Squared Error (RMSE) between the original non-zero ratings and the predicted ratings to evaluate the model's accuracy on the training data (for simplicity).

In [ ]:
# Flatten matrices to compare
original_flat = user_item_matrix.values.flatten()
predicted_flat = df_reconstructed.values.flatten()

# Filter only non-zero original ratings (we only care about accuracy on known ratings here)
mask = original_flat > 0
rmse = np.sqrt(mean_squared_error(original_flat[mask], predicted_flat[mask]))

print(f"Training RMSE: {rmse:.4f}")
print("Note: A lower RMSE indicates better fit, but beware of overfitting on small datasets.")